# 06: Representation collapse and variance-covariance regularization

![Representation geometry](../images/06_representation_collapse.svg)

**Learning goals:** recognize collapse, compute feature statistics, implement variance and covariance penalties, and compare token-level with pooled regularization. This notebook uses only deterministic synthetic data and runs on CPU.

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

SEED = 6
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
print(f'NumPy {np.__version__}, PyTorch {torch.__version__}, device=cpu')

## 1. Three representation geometries

Rows are observations and columns are features. A constant matrix is completely collapsed. A matrix whose columns copy one latent scalar has variation but only one useful direction. A healthy synthetic matrix varies along several directions.

In [ ]:
B, D = 128, 8
collapsed = np.ones((B, D))
latent = np.random.randn(B, 1)
redundant = latent @ np.linspace(0.5, 2.0, D)[None, :]
healthy = np.random.randn(B, D)

def summary(z):
    centered = z - z.mean(axis=0, keepdims=True)
    cov = centered.T @ centered / (len(z) - 1)
    return z.std(axis=0, ddof=1).mean(), np.linalg.matrix_rank(cov)

for name, z in [('collapsed', collapsed), ('redundant', redundant), ('healthy', healthy)]:
    mean_std, rank = summary(z)
    print(f'{name:10s} mean std={mean_std:.3f}, covariance rank={rank}')
assert summary(collapsed)[1] == 0
assert summary(redundant)[1] == 1

## 2. Vectorized losses

`var(axis=0, ddof=1)` computes one sample variance per feature. The covariance identity `X.T @ X / (B - 1)` computes all feature pairs in one optimized matrix multiplication. The variance hinge only penalizes standard deviations below the target. The covariance term removes the diagonal because feature scale is handled separately.

In [ ]:
def numpy_var_cov(z, target_std=1.0, eps=1e-4):
    if z.ndim != 2 or z.shape[0] < 2:
        raise ValueError('expected [observations, features] with at least two rows')
    x = z - z.mean(axis=0, keepdims=True)
    std = np.sqrt(x.var(axis=0, ddof=1) + eps)
    variance_loss = np.maximum(0.0, target_std - std).mean()
    cov = x.T @ x / (z.shape[0] - 1)
    off_diag = cov - np.diag(np.diag(cov))
    covariance_loss = (off_diag ** 2).sum() / z.shape[1]
    return variance_loss, covariance_loss, cov

for name, z in [('collapsed', collapsed), ('redundant', redundant), ('healthy', healthy)]:
    lv, lc, _ = numpy_var_cov(z)
    print(f'{name:10s} variance loss={lv:.3f}, covariance loss={lc:.3f}')
assert numpy_var_cov(collapsed)[0] > 0.9
assert numpy_var_cov(collapsed)[1] == 0.0

The constant matrix shows why both terms are required: its covariance penalty is zero, but its variance penalty is large. The redundant matrix shows the opposite case: it varies, yet its off-diagonal penalty is large.

In [ ]:
def torch_var_cov(z, target_std=1.0, eps=1e-4):
    x = z - z.mean(dim=0, keepdim=True)
    std = torch.sqrt(x.var(dim=0, correction=1) + eps)
    variance_loss = torch.relu(target_std - std).mean()
    cov = x.T @ x / (z.shape[0] - 1)
    mask = ~torch.eye(z.shape[1], dtype=torch.bool, device=z.device)
    covariance_loss = cov[mask].square().sum() / z.shape[1]
    return variance_loss, covariance_loss

z = torch.tensor(redundant, dtype=torch.float32, requires_grad=True)
lv, lc = torch_var_cov(z)
total = lv + 0.05 * lc
total.backward()
print(f'total={total.item():.3f}, gradient norm={z.grad.norm().item():.3f}')
assert torch.isfinite(z.grad).all()
# A Boolean mask avoids changing cov in place and keeps autograd behavior clear.

## 3. Token-level versus pooled statistics

For `[batch, tokens, features]`, `reshape(-1, D)` treats tokens as observations. `mean(dim=1)` instead creates one pooled observation per example. Flattening can reveal local collapse, but tokens within an example are correlated and should not be mistaken for independent samples.

In [ ]:
B, T, D = 32, 6, 8
base = torch.randn(B, 1, D)
offsets = torch.linspace(-1.0, 1.0, T).view(1, T, 1)
tokens = base + offsets
flat = tokens.reshape(B * T, D)
pooled = tokens.mean(dim=1)
tok_losses = torch_var_cov(flat)
pool_losses = torch_var_cov(pooled)
print('token losses:', tuple(round(x.item(), 3) for x in tok_losses))
print('pooled losses:', tuple(round(x.item(), 3) for x in pool_losses))
assert flat.shape == (B * T, D) and pooled.shape == (B, D)

In [ ]:
_, _, cov_bad = numpy_var_cov(redundant)
_, _, cov_good = numpy_var_cov(healthy)
fig, axes = plt.subplots(1, 2, figsize=(7, 2.8), constrained_layout=True)
for ax, cov, title in zip(axes, [cov_bad, cov_good], ['Redundant', 'Healthy']):
    image = ax.imshow(cov, cmap='coolwarm', vmin=-2, vmax=2)
    ax.set(title=title, xlabel='feature', ylabel='feature')
fig.colorbar(image, ax=axes, shrink=0.8, label='covariance')
plt.show()

## Exercises and takeaways

1. Change `target_std` to 0.5. Which matrices change their variance penalty? **Check:** only features below the new threshold are penalized.
2. Standardize `redundant` featurewise before computing covariance. **Check:** its off-diagonal entries remain near plus or minus one because scaling cannot remove exact dependence.
3. Construct token pairs `u` and `-u`. **Check:** pooled variance collapses to zero while token-level variance remains positive.

**Takeaways:** prediction agreement can accept constant solutions; variance prevents constant coordinates; covariance discourages copied coordinates; and token and pooled statistics answer different questions. Full covariance costs `O(B D^2)` time and `O(D^2)` memory, so feature blocks can help when `D` is very large.

## Continue learning

[Previous notebook: 05](05_masked_latent_prediction.ipynb) | [Lecture](../lectures/06_representation_collapse.md) | [Curriculum](../README.md) | [Next notebook: 07](07_gradient_updates_and_schedules.ipynb)